# Cohere Reranker — 프로덕션급 클라우드 재순위화

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- **Cohere Rerank API**의 특징과 Cross-Encoder(로컬) 대비 장단점 이해하기
- `CohereEmbeddings`와 `CohereRerank`를 결합한 검색 파이프라인 구현하기
- API 키와 모델 선택 방법 익히기

## 사전 지식

- 01-Cross-Encoder-Reranker에서 Two-Stage Retrieval 개념
- Cohere API 키 발급 방법 (`COHERE_API_KEY` 환경변수 필요)

---

```mermaid
flowchart LR
    Q[사용자 쿼리]:::input --> CE[Cohere Embeddings<br/>초기 벡터 검색<br/>k=10]:::process
    CE --> CR[Cohere Rerank API<br/>cloud 재정렬<br/>top_n=3]:::external
    CR --> R[관련성 점수 포함<br/>상위 3개 문서]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

## Cohere vs Cross-Encoder (로컬) 비교

| 특징 | Cohere Reranker | Cross-Encoder (로컬) |
|------|:---:|:---:|
| 정확도 | 최고 수준 | 높음 |
| 다국어 | 100+ 언어 | 모델 의존 |
| 비용 | API 사용료 | 인프라 비용 |
| 설정 | 즉시 사용 | 모델 다운로드 필요 |
| 추천 용도 | 프로덕션 서비스 | 개발·테스트 |

> **실무 팁**: `COHERE_API_KEY`가 없어도 노트북의 핵심 개념을 확인할 수 있어요. API 키 없이도 코드 흐름을 이해하고 실습해보세요.

## 1. 환경 설정

### 1.1 Cohere API 키 발급

1. [Cohere 웹사이트](https://dashboard.cohere.com/api-keys)에서 API 키 발급
2. `.env` 파일에 API 키 추가:

```bash
COHERE_API_KEY="your-cohere-api-key"
```

### 1.2 Cohere 모델 정보

**Embedding 모델**:
- `embed-multilingual-v3.0`: 최신 다국어 임베딩 (권장)
- `embed-multilingual-light-v3.0`: 경량 버전
- `embed-multilingual-v2.0`: 이전 버전

**Reranker 모델**:
- `rerank-multilingual-v3.0`: 최신 다국어 재정렬 (권장)
- `rerank-english-v3.0`: 영어 특화 모델
- `rerank-multilingual-v2.0`: 이전 버전

In [ ]:
# 필요한 라이브러리 import
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_cohere import CohereEmbeddings, CohereRerank
from langchain.retrievers import ContextualCompressionRetriever

# 환경 변수 로드

# 아래에 코드를 작성하세요


## 2. 문서 출력 도우미 함수

In [ ]:
def pretty_print_docs(docs, show_scores=True):
    """검색된 문서를 보기 좋게 출력하는 함수"""
    print(f"\n{'=' * 100}")
    print(f"총 {len(docs)}개 문서 검색됨")
    print(f"{'=' * 100}\n")
    
    for i, doc in enumerate(docs, 1):
        print(f"[문서 {i}]")
        
        # Cohere Reranker는 relevance_score를 metadata에 추가
        if show_scores and 'relevance_score' in doc.metadata:
            score = doc.metadata['relevance_score']
            print(f"관련성 점수: {score:.4f}")
        
        print(f"내용: {doc.page_content}")
        print(f"{'─' * 100}\n")


## 3. Cohere Embeddings로 벡터 검색 시스템 구축

Cohere의 다국어 임베딩 모델을 사용하여 초기 검색 시스템을 구축합니다.

In [ ]:
# 1단계: 문서 로드

# 아래에 코드를 작성하세요


In [ ]:
# 2단계: 텍스트 분할

# 아래에 코드를 작성하세요


In [ ]:
# ---------------------------------------------------
# Cohere 임베딩으로 FAISS 벡터스토어 생성
# ---------------------------------------------------
# ============================================================
# TODO: CohereEmbeddings와 FAISS로 벡터스토어를 만들고 검색기를 설정하세요
# 힌트: CohereEmbeddings(model="embed-multilingual-v3.0"), k=10
# 예상 결과: "Cohere 벡터스토어 생성 완료" 출력
# ============================================================

# 3단계: Cohere 임베딩 모델 초기화
    # embed-multilingual-v3.0: Cohere의 최신 다국어 임베딩 모델 (한국어 포함)

    # 4단계: FAISS 벡터스토어 생성 및 검색기 설정


In [ ]:
# ---------------------------------------------------
# Reranker 적용 전 기본 검색 결과 확인
# ---------------------------------------------------
# ============================================================
# TODO: base_retriever로 쿼리를 검색하고 결과를 출력하세요
# 힌트: invoke(query) 호출, show_scores=False로 점수 숨기기
# 예상 결과: 10개 문서가 Cohere 임베딩 유사도 순으로 출력됨
# ============================================================

# 기본 검색 테스트


    # Reranker 적용 전 기본 검색


## 4. Cohere Reranker 적용

Cohere의 Rerank API를 사용하여 검색 결과를 재정렬합니다.

In [ ]:
# ---------------------------------------------------
# Cohere Rerank API로 압축 검색기 구성
# ---------------------------------------------------
# ============================================================
# TODO: CohereRerank와 ContextualCompressionRetriever로 재정렬 파이프라인을 만드세요
# 힌트: CohereRerank(model="rerank-multilingual-v3.0", top_n=3)
# 예상 결과: "Cohere Reranker 설정 완료" 출력
# ============================================================

    # 1단계: Cohere Reranker 초기화
    # rerank-multilingual-v3.0: 최신 다국어 재정렬 모델 (100+ 언어 지원)

    # 2단계: 압축 검색기 생성


In [ ]:
# ---------------------------------------------------
# Reranker 적용 후 검색 결과 확인
# ---------------------------------------------------
# ============================================================
# TODO: compression_retriever로 같은 쿼리를 검색하고 관련성 점수를 확인하세요
# 힌트: invoke(query) 후 pretty_print_docs(docs, show_scores=True)
# 예상 결과: 3개 문서와 각각의 Cohere 관련성 점수 출력
# ============================================================

    # Reranker 적용 검색


## 5. 결과 비교 및 분석

In [ ]:
# 아래에 코드를 작성하세요


## 6. 다양한 쿼리 테스트

In [ ]:
# ---------------------------------------------------
# 다양한 쿼리로 Cohere Reranker 성능 검증
# ---------------------------------------------------
# ============================================================
# TODO: 여러 쿼리를 순회하며 Cohere Reranker로 검색하고 관련성 점수를 비교하세요
# 힌트: for 루프로 test_queries 순회, compression_retriever.invoke(test_query)
# 예상 결과: 각 쿼리별 상위 3개 문서와 Cohere 관련성 점수 출력
# ============================================================

    # 다양한 쿼리로 Cohere Reranker 성능 검증


        
        # Cohere Reranker 적용


## 7. 핵심 정리

### Reranker 비교

| 특징 | Cohere Reranker | Cross-Encoder (로컬) |
|------|:---:|:---:|
| 정확도 | 최고 수준 | 높음 |
| 다국어 | 100+ 언어 | 모델 의존 |
| 비용 | API 사용료 | 무료 (인프라 비용) |
| 관리 | 완전 관리형 | 직접 관리 |
| 추천 용도 | 프로덕션 서비스 | 개발/테스트 |

### 기본 사용법

```python
from langchain_cohere import CohereRerank

compressor = CohereRerank(
    model="rerank-multilingual-v3.0",
    top_n=3,
)
```

### 주요 파라미터

| 파라미터 | 설명 | 기본값 |
|---------|------|--------|
| `model` | Reranker 모델명 | `rerank-multilingual-v3.0` |
| `top_n` | 최종 반환 문서 수 | 3 |
| `cohere_api_key` | Cohere API 키 | 환경변수 `COHERE_API_KEY` |

> **실무 팁**: Cohere는 무료 체험 크레딧을 제공합니다. 먼저 무료 API 키로 테스트하고, 실제 트래픽이 발생할 때 유료 플랜으로 전환하는 것을 권장해요.

## 마무리 정리

이 노트북에서 배운 내용을 정리해요:

### Reranker 선택 기준

| 상황 | 추천 |
|------|------|
| 개발·테스트 환경 | Cross-Encoder (로컬, 무료) |
| 한국어 프로덕션 서비스 | **Cohere** (다국어, 관리형) |
| 긴 문서(2K+ 토큰) | Jina Reranker (8K 지원) |
| 완전한 데이터 통제 | Cross-Encoder (온프레미스) |

### 다음 노트북 예고

**03-Jina-Reranker**: 최대 8K 토큰까지 처리 가능한 Jina Reranker를 배워요. 긴 문서 재순위화에서 차별화된 성능을 확인해볼 수 있어요.